# v6 — Geração de dados/figuras/tabelas para o relatório (Cap. 3)

Notebook único que reproduz o pipeline final e gera, de forma limpa, as tabelas e
figuras do Capítulo 3. Segue o `ROTEIRO_CAP3.md`. Cada seção corresponde a uma
subseção do relatório.

## Setup, dataset e extração de features

As features são organizadas em **grupos** para permitir o refinamento incremental (3.4).
O grupo `base` é o conjunto inicial usado no benchmark da seção 3.3.

In [ ]:
import json
import urllib.request
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from skimage.feature import local_binary_pattern, hog
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from xgboost import XGBClassifier
from tqdm import tqdm

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "InfraredSolarModules"
DATASET_URL = "https://github.com/RaptorMaps/InfraredSolarModules/raw/master/2020-02-14_InfraredSolarModules.zip"


def garantir_dataset():
    if (DATA_DIR / "module_metadata.json").exists():
        return
    zip_path = BASE_DIR / "2020-02-14_InfraredSolarModules.zip"
    if not zip_path.exists():
        urllib.request.urlretrieve(DATASET_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(BASE_DIR)


garantir_dataset()
df = pd.DataFrame.from_dict(json.load(open(DATA_DIR / "module_metadata.json")), orient="index")

GABOR_KERNELS = [cv2.getGaborKernel((9, 9), 2.0, th, 4.0, 0.5, 0, ktype=cv2.CV_32F)
                 for th in np.deg2rad([0, 45, 90, 135])]


def extrair_features(img):
    img = img.astype(np.uint8)
    total = img.size
    f = {}
    # --- base (conjunto inicial, secao 3.3) ---
    f["mean_int"] = float(img.mean())
    f["std_int"] = float(img.std())
    f["max_int"] = float(img.max())
    f["min_int"] = float(img.min())
    f["p90_int"] = float(np.percentile(img, 90))
    otsu_t, _ = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, hot = cv2.threshold(img, max(otsu_t, 180), 255, cv2.THRESH_BINARY)
    hot = cv2.morphologyEx(hot, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
    f["hot_fraction"] = float(hot.sum() / 255) / total
    contornos, _ = cv2.findContours(hot, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    blobs = [c for c in contornos if cv2.contourArea(c) >= 2]
    f["num_blobs"] = float(len(blobs))
    if blobs:
        maior = max(blobs, key=cv2.contourArea)
        area = cv2.contourArea(maior)
        x, y, w, h = cv2.boundingRect(maior)
        f["largest_area"] = float(area)
        f["largest_extent"] = float(area / (w * h)) if w * h > 0 else 0.0
        f["largest_aspect"] = float(w / h) if h > 0 else 0.0
    else:
        f["largest_area"] = f["largest_extent"] = f["largest_aspect"] = 0.0
    f["row_cov"] = float((hot.sum(axis=1) > 0).mean())
    f["col_cov"] = float((hot.sum(axis=0) > 0).mean())
    f["dark_fraction"] = float((img < (img.mean() - img.std())).mean())
    f["edge_density"] = float((cv2.Canny(img, 50, 150) > 0).mean())
    gx = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=3)
    f["grad_mean"] = float(np.sqrt(gx ** 2 + gy ** 2).mean())
    imgf = img.astype(np.float32)
    f["sym_lr"] = float(np.abs(imgf - np.fliplr(imgf)).mean())
    f["sym_tb"] = float(np.abs(imgf - np.flipud(imgf)).mean())
    hist = cv2.calcHist([img], [0], None, [8], [0, 256]).flatten()
    hist = hist / hist.sum()
    for i, hv in enumerate(hist):
        f[f"hist{i}"] = float(hv)
    # --- grupos adicionais (secao 3.4) ---
    f["skew_int"] = float(skew(imgf.ravel())) if img.std() > 0 else 0.0
    f["kurt_int"] = float(kurtosis(imgf.ravel())) if img.std() > 0 else 0.0
    hu = np.zeros(7)
    if blobs:
        mk = np.zeros_like(img)
        cv2.drawContours(mk, [maior], -1, 255, -1)
        huv = cv2.HuMoments(cv2.moments(mk)).flatten()
        hu = np.array([-np.sign(v) * np.log10(abs(v) + 1e-30) for v in huv])
    for i in range(7):
        f[f"hu_{i}"] = float(hu[i])
    hh, ww = img.shape
    hs, ws = hh // 3, ww // 3
    for i in range(3):
        for j in range(3):
            y0, y1 = i * hs, (hh if i == 2 else (i + 1) * hs)
            x0, x1 = j * ws, (ww if j == 2 else (j + 1) * ws)
            f[f"grid_{i}{j}"] = float(img[y0:y1, x0:x1].mean())
    lbp = local_binary_pattern(img, P=8, R=1, method="uniform")
    lh, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)
    for k, v in enumerate(lh):
        f[f"lbp_{k}"] = float(v)
    for i, kern in enumerate(GABOR_KERNELS):
        resp = cv2.filter2D(imgf, cv2.CV_32F, kern)
        f[f"gabor{i}_mean"] = float(resp.mean())
        f[f"gabor{i}_std"] = float(resp.std())
    for k, v in enumerate(hog(imgf, orientations=8, pixels_per_cell=(8, 8),
                              cells_per_block=(1, 1), feature_vector=True, channel_axis=None)):
        f[f"hog_{k}"] = float(v)
    return f


# Nomes do grupo base (conjunto inicial da secao 3.3)
BASE_FEATS = ["mean_int", "std_int", "max_int", "min_int", "p90_int", "hot_fraction",
              "num_blobs", "largest_area", "largest_extent", "largest_aspect", "row_cov",
              "col_cov", "dark_fraction", "edge_density", "grad_mean", "sym_lr", "sym_tb",
              "hist0", "hist1", "hist2", "hist3", "hist4", "hist5", "hist6", "hist7"]

registros, y = [], []
for _, row in tqdm(df.iterrows(), total=df.shape[0]):
    img = cv2.imread(str(DATA_DIR / row["image_filepath"]), cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    registros.append(extrair_features(img))
    y.append(row["anomaly_class"])
y = np.array(y)
FEAT_NAMES = list(registros[0].keys())
X_full = np.array([[r[k] for k in FEAT_NAMES] for r in registros])
print(f"Total de imagens: {len(y)} | features extraidas: {len(FEAT_NAMES)} | base: {len(BASE_FEATS)}")

# Split unico reutilizado em todas as secoes
idx = np.arange(len(y))
idx_tr, idx_te = train_test_split(idx, test_size=0.3, random_state=42, stratify=y)
y_tr, y_te = y[idx_tr], y[idx_te]


classes = sorted(set(y))


def treinar_e_avaliar(modelo, X, nome):
    """Treina nas linhas de treino e avalia no teste; retorna (resumo, predicoes)."""
    Xtr, Xte = X[idx_tr], X[idx_te]
    if nome == "XGBoost":
        le = LabelEncoder().fit(y)
        modelo.fit(Xtr, le.transform(y_tr), sample_weight=compute_sample_weight("balanced", y_tr))
        pred = le.inverse_transform(modelo.predict(Xte))
    elif nome == "Gradient Boosting":
        modelo.fit(Xtr, y_tr, sample_weight=compute_sample_weight("balanced", y_tr))
        pred = modelo.predict(Xte)
    else:
        modelo.fit(Xtr, y_tr)
        pred = modelo.predict(Xte)
    resumo = {"modelo": nome,
              "acuracia": round(accuracy_score(y_te, pred) * 100, 1),
              "f1_macro": round(f1_score(y_te, pred, average="macro") * 100, 1)}
    return resumo, pred

## 3.3 Benchmark de classificadores (conjunto base de features)

Compara RandomForest, Gradient Boosting, SVM e XGBoost no mesmo conjunto inicial de
features e mesmo split. Todos com tratamento de desbalanceamento.

In [ ]:
X_base = X_full[:, [FEAT_NAMES.index(f) for f in BASE_FEATS]]

modelos_33 = [
    (RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1), "RandomForest"),
    (HistGradientBoostingClassifier(random_state=42), "Gradient Boosting"),
    (make_pipeline(StandardScaler(), SVC(class_weight="balanced", random_state=42)), "SVM"),
    (XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.3, tree_method="hist",
                   random_state=42, n_jobs=-1, eval_metric="mlogloss"), "XGBoost"),
]

# Siglas para as colunas
SIGLA = {"RandomForest": "RF", "Gradient Boosting": "GB", "SVM": "SVM", "XGBoost": "XGB"}
ordem_mod = ["RandomForest", "Gradient Boosting", "SVM", "XGBoost"]

resumos_33, prf_33 = [], {}
for m, n in modelos_33:
    resumo, pred = treinar_e_avaliar(m, X_base, n)
    resumos_33.append(resumo)
    p, r, f, _ = precision_recall_fscore_support(y_te, pred, labels=classes, zero_division=0)
    prf_33[n] = {"P": p * 100, "R": r * 100, "F1": f * 100}

bench_33 = pd.DataFrame(resumos_33).set_index("modelo").sort_values("f1_macro", ascending=False)
print("\n=== 3.3 Benchmark geral (features base) ===")
print(bench_33)
print("\n--- LaTeX (geral) ---")
for nome, rr in bench_33.iterrows():
    print(f"{SIGLA[nome]} & {rr['acuracia']:.1f}\\% & {rr['f1_macro']:.1f}\\% \\\\")

# Precision / Recall / F1 por classe; metrica no topo, modelos (RF/GB/SVM/XGB) como subcolunas
print("\n--- LaTeX (por classe: metrica > modelo, %) ---")
for ci, cls in enumerate(classes):
    cells = []
    for metrica in ["P", "R", "F1"]:
        for n in ordem_mod:
            cells.append(f"{prf_33[n][metrica][ci]:.0f}")
    print(f"{cls} & " + " & ".join(cells) + r" \\")